# Condition C0 — Baseline (no examples)

Flan-T5 base + large, question + serialized table only. Sets the floor for every other condition. Metric: Exact-Match + token-F1 on WikiTableQuestions.

In [ ]:
# --- Setup ---
# On Colab: clone the repo and install deps INTO THE KERNEL (%pip, not !pip).
# Locally: find the repo root and put it on sys.path. No clone, no install --
# run this notebook with the project's .venv kernel ("ECS111 (.venv)"), which
# already has datasets/transformers/torch. Locally this is a true no-op.
import os
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir("ECS111FinalProject"):
        !git clone https://github.com/adiseshvsanklapur/ECS111FinalProject.git
    os.chdir("ECS111FinalProject")
    %pip -q install -r requirements.txt
else:
    # Walk up from the notebook's directory to the repo root (the dir with src/).
    root = os.path.abspath(os.getcwd())
    while root != os.path.dirname(root) and not os.path.isdir(os.path.join(root, "src")):
        root = os.path.dirname(root)
    if not os.path.isdir(os.path.join(root, "src")):
        raise RuntimeError(
            "Could not find the repo root (no src/ found walking up). "
            "Open this notebook from inside the cloned ECS111FinalProject."
        )
    os.chdir(root)
    if root not in sys.path:
        sys.path.insert(0, root)

print("cwd:", os.getcwd())

In [ ]:
# Set SMOKE = False for the full, reported run. SMOKE = True does a fast real
# end-to-end pass (flan-t5-small, tiny slice) to confirm everything works first.
SMOKE = True

In [ ]:
from src import config
device = config.get_device()
print("device:", device)

if SMOKE:
    prompt_models = [config.SMOKE_MODEL]
    seeds = [13]
    eval_n = config.SMOKE_EVAL_N
    eval_n_cot = config.SMOKE_EVAL_N
    train_n = config.SMOKE_TRAIN_N
else:
    prompt_models = config.PROMPT_MODELS      # flan-t5-base + large
    seeds = config.SEEDS                       # [13, 42]
    eval_n = config.EVAL_N                      # 1000
    eval_n_cot = config.EVAL_N_COT             # 500
    train_n = config.TRAIN_N                    # 8000

In [ ]:
from src.data import load_wtq_eval
from src.prompts import build_baseline_prompt
from src.evaluate import predict_and_evaluate
from src.trainer import load_model_and_tokenizer

rows = []
for model_id in prompt_models:
    for seed in seeds:
        examples = load_wtq_eval(n=eval_n, seed=seed)
        model, tok, device = load_model_and_tokenizer(model_id, device)
        res = predict_and_evaluate(
            model, tok, examples, build_baseline_prompt,
            condition="baseline", model_id=model_id, seed=seed,
            task="wtq", device=device,
        )
        print(model_id, "seed", seed, res["metrics"])
        rows.append((model_id, seed, res["metrics"]))
rows